# Modelo Data Challenge

In [ ]:
import os
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2

### 1. Extraindo features e criando dataframe (.csv)

In [ ]:
from Criar_DataFrame import executar_extracao
from Dataframes import dataframe_train, dafaframe_test

#### O primeiro bloco é para criar os arquivos (leva um tempo). O segundo é para apenas ler os arquivos já criados, caso tenha que refazer.
#### Eu altero entre eles transformando um dos dois blobos em comentário.

In [ ]:
# print("=== EXTRAINDO DADOS DE TREINO ===")
# portas_treino = dataframe_train()

# print("\n=== EXTRAINDO DADOS DE TESTE ===")
# portas_teste = dafaframe_test()

# pasta_destino = "Dados_Processados"
# if not os.path.exists(pasta_destino): 
#     os.makedirs(pasta_destino)

# # 1. Empilha as listas de DataFrames em um DataFrame único gigante
# df_treino_completo = pd.concat([df for porta_id, df in portas_treino], ignore_index=True)
# df_teste_completo = pd.concat([df for porta_id, df in portas_teste], ignore_index=True)

# # 2. Salva em formato CSV
# caminho_treino = os.path.join(pasta_destino, "treino_features.csv")
# caminho_teste = os.path.join(pasta_destino, "teste_features.csv")

# df_treino_completo.to_csv(caminho_treino, index=False) # index=False evita criar uma coluna inútil de numeração
# df_teste_completo.to_csv(caminho_teste, index=False)
    
# print("\n✅ Extração dupla concluída e salva em formato CSV!")

In [ ]:
print("Carregando bases de dados em CSV...")
df_treino_csv = pd.read_csv('Dados_Processados/treino_features.csv')
df_teste_csv = pd.read_csv('Dados_Processados/teste_features.csv')

# ========================================================
# O  DE SALVAMENTO: Recriando o Porta_ID caso ele não exista!
# Toda vez que o 'Ciclo' cai (ex: vai de 150 de volta para 1), ele soma +1 no ID.
# ========================================================
if 'Porta_ID' not in df_treino_csv.columns:
    print("Recriando a coluna Porta_ID no Treino...")
    df_treino_csv['Porta_ID'] = (df_treino_csv['Ciclo'] < df_treino_csv['Ciclo'].shift(1)).cumsum() + 1

if 'Porta_ID' not in df_teste_csv.columns:
    print("Recriando a coluna Porta_ID no Teste...")
    df_teste_csv['Porta_ID'] = (df_teste_csv['Ciclo'] < df_teste_csv['Ciclo'].shift(1)).cumsum() + 1

# 2. O truque: o groupby converte o DataFrame gigante de volta para a estrutura: [(1, df_porta1), ...]
portas_treino = list(df_treino_csv.groupby('Porta_ID'))
portas_teste = list(df_teste_csv.groupby('Porta_ID'))

print(f"✅ Sucesso! Encontradas {len(portas_treino)} portas de Treino e {len(portas_teste)} portas de Teste.")

### 2. Mostra um gráfico com as curvas de degradação das 48 Train's

In [ ]:
# from Criar_Grafico import gera_grafico

# gera_grafico(portas_teste)

### 3. Modelo

In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV

X_train_list, y_train_list = [], []

# ====================================================================
# PASSO 1: PREPARAÇÃO DOS DADOS DE TREINO
# ====================================================================
# Lista unificada de colunas para remover do modelo (Evita o erro de Feature Missing!)
colunas_para_ignorar = ['Porta_ID', 'Ciclo', 'Ciclo_Relativo', 'RUL_Gabarito']

for porta_id, df_porta in portas_treino:
    df_features = df_porta.copy() 
    
    # 1. Calcula o RUL real (O alvo do modelo)
    if 'RUL_Gabarito' in df_features.columns:
        target = df_features['RUL_Gabarito']
    else:
        ciclo_da_falha = df_features['Ciclo'].max()
        target = ciclo_da_falha - df_features['Ciclo']
    
    # 2. Removemos a resposta e os relógios. O modelo foca apenas nos sensores!
    features = df_features.drop(columns=colunas_para_ignorar, errors='ignore')
    
    X_train_list.append(features)
    y_train_list.append(target)

# Junta todas as portas numa única matriz de aprendizagem
X_train_full = pd.concat(X_train_list, ignore_index=True).fillna(0)
y_train_full = np.concatenate(y_train_list)

print("A ajustar os hiperparâmetros (Cross-Validation) e treinando...")
modelo_lasso = LassoCV(alphas=np.logspace(-6, 1, 100), cv=10, max_iter=50000)
modelo_lasso.fit(X_train_full, y_train_full)
print("✅ Modelo treinado com sucesso!")

# ====================================================================
# PASSO 2: PREPARAÇÃO DOS DADOS DE TESTE
# ====================================================================
X_test_list = []

for porta_id, df_porta in portas_teste:
    df_features_teste = df_porta.copy() 
    
    # Removemos EXATAMENTE as mesmas colunas do treino
    features_teste = df_features_teste.drop(columns=colunas_para_ignorar, errors='ignore')
    
    X_test_list.append(features_teste)

# Junta todos os testes numa única matriz
X_test_full = pd.concat(X_test_list, ignore_index=True).fillna(0)

# ====================================================================
# PASSO 3: A PREDIÇÃO
# ====================================================================
print("Realizando as predições na base de Teste...")
RUL_predito = modelo_lasso.predict(X_test_full)
print("✅ Predições concluídas!")

# ====================================================================
# PASSO 4: MONTANDO O DATAFRAME DE RESULTADOS
# ====================================================================
ids_teste = []
ciclos_teste = []

for porta_id, df in portas_teste:
    # Pegamos a variável externa 'porta_id' e multiplicamos pelo tamanho da tabela!
    ids_teste.extend([porta_id] * len(df))
    ciclos_teste.extend(df['Ciclo'].tolist())

df_resultados = pd.DataFrame({
    'Porta_ID': ids_teste,
    'Ciclo_Atual': ciclos_teste,
    'RUL_Final': RUL_predito
})

A ajustar os hiperparâmetros (Cross-Validation) e treinando...
✅ Modelo treinado com sucesso!
Realizando as predições na base de Teste...
✅ Predições concluídas!


In [ ]:
# import pickle
# import pandas as pd

# # 1. Importações do seu novo arquivo 'modelo.py'
# from modelo import extrair_dados_com_memoria, pipeline_predicao_ciclo_a_ciclo

# # 2. Importações do seu arquivo de formatação
# from SubmissionFormat import gera_arquivo_datachallenge, gera_arquivo_gabarito

# # ==============================================================================
# # FASE 1: EXTRAÇÃO DE DADOS (Agora com a memória física do passado)
# # ==============================================================================
# portas_treino = extrair_dados_com_memoria(tipo="Train", num_portas=48)
# portas_teste = extrair_dados_com_memoria(tipo="Test", num_portas=19)


# # ==============================================================================
# # FASE 2: TREINO E SIMULAÇÃO DE TEMPO REAL
# # ==============================================================================
# # Esta única função treina o modelo e já te devolve a tabela de histórico pronta!
# modelo_treinado, df_resultados = pipeline_predicao_ciclo_a_ciclo(portas_treino, portas_teste)




In [ ]:
# # ==============================================================================
# # FASE 3: EXPORTAÇÃO
# # ==============================================================================
# # Renomeia a coluna para o padrão que o seu SubmissionFormat.py espera
# df_resultados = df_resultados.rename(columns={'RUL_Predito': 'RUL_Final'})

# # Gera os arquivos finais
# gera_arquivo_datachallenge(df_resultados)
# gera_arquivo_gabarito(df_resultados)

# print("\n✅ Fluxo principal concluído com sucesso!")

In [18]:
import pandas as pd

# 1. Criar o DataFrame inicial com os teus resultados
df_submissao = pd.DataFrame({
    'Porta_ID': ids_teste,
    'RUL_Pred': RUL_predito
})

# 2. Tratamento de Segurança:
# - Garante que o RUL não é negativo (mínimo 1)
# - Arredonda para números inteiros
df_submissao['RUL_Pred'] = df_submissao['RUL_Pred'].clip(lower=1)
df_submissao['RUL_Pred'] = df_submissao['RUL_Pred'].round(0).astype(int)

# 3. Formatação para a Competição (Duplicar linhas):
# Cada ciclo de teste tem dois ficheiros (Opening e Closing). 
# Precisamos de repetir cada previsão duas vezes no CSV.
df_final = df_submissao[['Porta_ID', 'RUL_Pred']]
df_final = df_final.loc[df_final.index.repeat(2)].reset_index(drop=True)

# 4. Salvar em .csv
# Formato: sem cabeçalho, sem índice, separado por ponto e vírgula
nome_arquivo = 'submission.csv'

try:
    df_final.to_csv(nome_arquivo, sep=';', header=False, index=False)
    print(f"✅ Ficheiro '{nome_arquivo}' gerado com sucesso!")
    print(f"Total de linhas: {len(df_final)}")
except PermissionError:
    print(f"❌ Erro: O ficheiro '{nome_arquivo}' está aberto noutro programa (como o Excel). Fecha-o e tenta novamente.")

✅ Ficheiro 'submission.csv' gerado com sucesso!
Total de linhas: 47178


### 5. Score e tabela com valores

In [17]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ========================================================
# FUNÇÃO DE NORMALIZAÇÃO MATEMÁTICA
# ========================================================
def norm_f(m, a=4443.76, b=1.53, c=4443.76, d=0):
    return (a / ((m**b) + c)) + d


print("\nCalculando o Score Oficial diretamente das arrays de predição...")

# ========================================================
# 1. EXTRAÇÃO DO GABARITO E MONTAGEM DA TABELA
# ========================================================
RUL_gabarito = []

# Extraímos a verdade absoluta (gabarito) direto da lista de teste
for porta_id, df in portas_teste:
    if 'RUL_Gabarito' in df.columns:
        RUL_gabarito.extend(df['RUL_Gabarito'].tolist())
    else:
        ciclo_final = df['Ciclo'].max()
        RUL_gabarito.extend((ciclo_final - df['Ciclo']).tolist())

# Montamos a tabela mestre com tudo alinhado
df_completo = pd.DataFrame({
    'Porta_ID': ids_teste,
    'Ciclo_Atual': ciclos_teste,
    'RUL_Pred': RUL_predito,
    'RUL_Real': RUL_gabarito
})


# ========================================================
# 2. CÁLCULO DAS MÉTRICAS POR PORTA
# ========================================================
alpha_peso = 2.0
resultados_viagem = []

# Iteramos sobre as portas da tabela unificada
for porta in df_completo['Porta_ID'].unique():
    
    df_porta = df_completo[df_completo['Porta_ID'] == porta].sort_values('Ciclo_Atual').copy()
    
    T = len(df_porta) 
    if T == 0: 
        continue
        
    y_true = df_porta['RUL_Real']
    y_pred = df_porta['RUL_Pred']
    
    erro = y_true - y_pred
    
    # --- RMSE ---
    erro_quadrado = erro ** 2 
    somatorio = np.sum(erro_quadrado)
    mse = somatorio / T
    rmse = np.sqrt(mse)

    # --- Precision ---
    erro_modulo = abs(erro) 
    
    # Pequena proteção matemática caso algum y_true chegue a zero
    fracao = erro_modulo / np.where(y_true == 0, 1e-9, y_true)
    
    fracoes_validas = fracao[fracao < 0.1]
    somatorio_prec = np.sum(fracoes_validas) if not fracoes_validas.empty else 0
    precicion = 100*(somatorio_prec / T)

    # --- NORMALIZAÇÃO ---
    precicion_norm = norm_f(precicion)
    rmse_norm = norm_f(rmse)

    # --- PROGNOSTIC HORIZON (PH INDIVIDUAL ROBUSTO) ---
    t_eof = df_porta['Ciclo_Atual'].max() 
    
    # Suaviza a linha de predição em uma janela de 5 ciclos
    rul_suavizada = df_porta['RUL_Pred'].rolling(window=5, min_periods=1).mean()
    
    margem_constante = t_eof * 0.10
    limite_inferior = df_porta['RUL_Real'] - margem_constante
    limite_superior = df_porta['RUL_Real'] + margem_constante
    
    dentro_dos_limites = (rul_suavizada >= limite_inferior) & (rul_suavizada <= limite_superior)
    fora_dos_limites = ~dentro_dos_limites 
    
    if fora_dos_limites.any():
        ultimo_erro = df_porta.loc[fora_dos_limites, 'Ciclo_Atual'].max()
        df_reta_final = df_porta[df_porta['Ciclo_Atual'] > ultimo_erro]
        
        if df_reta_final.empty:
            t_alpha = t_eof 
        else:
            t_alpha = df_reta_final['Ciclo_Atual'].min()
    else:
        t_alpha = df_porta['Ciclo_Atual'].min()
            
    ph = (t_eof - t_alpha) / t_eof if t_eof > 0 else 0

    # --- SCORE FINAL ---
    score = (rmse_norm + precicion_norm + (alpha_peso * ph)) / (2 + alpha_peso)

    resultados_viagem.append({
        'Porta_ID': porta,
        'T_Pontos': T, 
        'Precision_Bruto': round(precicion, 2),
        'Precision_Norm': round(precicion_norm, 4),
        'RMSE_Bruto': round(rmse, 2),
        'RMSE_Norm': round(rmse_norm, 4),
        'PH': round(ph, 4),
        'Score': round(score, 4)
    })

df_metricas = pd.DataFrame(resultados_viagem)

# ========================================================
# 3. TABELA INTERATIVA COM PLOTLY
# ========================================================
if not df_metricas.empty:
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=[f"<b>{col}</b>" for col in df_metricas.columns], 
            fill_color='#1f77b4', 
            font=dict(color='white', size=13),
            align='center',
            height=35
        ),
        cells=dict(
            values=[df_metricas[col] for col in df_metricas.columns],
            fill_color='#f5f6fa', 
            font=dict(color='black', size=12),
            align='center',
            height=30
        )
    )])

    fig.update_layout(
        title=dict(text='<b>Score Oficial do Modelo (Avaliação Direta na Memória)</b>', x=0.5, font=dict(size=18)),
        margin=dict(l=20, r=20, t=50, b=20), 
        height=400 
    )

    fig.show()
    
    # Imprime a média geral
    media_score = df_metricas['Score'].mean()
    print(f"\n🏆 SCORE GLOBAL MÉDIO: {media_score:.4f}")
else:
    print("\n❌ Nenhuma métrica foi calculada. Verifique os dados das predições.")


Calculando o Score Oficial diretamente das arrays de predição...



🏆 SCORE GLOBAL MÉDIO: 0.2767


### 6. Comparação da degração: Modelo vs Real

In [ ]:
# Importe a nova função do arquivo que acabamos de criar
from Comparacao import comparar_submissao_vs_gabarito

# 2. Chama o painel de auditoria visual passando as previsões E os dados brutos de teste
comparar_submissao_vs_gabarito()